In [1]:
!find / -name colab_data.csv 2>/dev/null
!pwd

/colab_data.csv
/content


In [2]:
!mv /colab_data.csv /content/
!ls /content

colab_data.csv	sample_data


In [3]:
!pip install -q sentence-transformers
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("colab_data.csv").dropna(subset=["text"])
train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
print(len(train), len(test))

63830 15958


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")
model.max_seq_length = 256

Xb_train = model.encode(train["text"].tolist(), batch_size=128, show_progress_bar=True)
Xb_test = model.encode(test["text"].tolist(), batch_size=128, show_progress_bar=True)
print(Xb_train.shape, Xb_test.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/499 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

(63830, 384) (15958, 384)


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(Xb_train, train["label"])
pred = clf.predict(Xb_test)

print("BERT-style embeddings + LogReg  acc:", round(accuracy_score(test["label"], pred), 4),
      " macro-F1:", round(f1_score(test["label"], pred, average="macro"), 4))
print(classification_report(test["label"], pred, target_names=["negative", "positive"]))

BERT-style embeddings + LogReg  acc: 0.8252  macro-F1: 0.723
              precision    recall  f1-score   support

    negative       0.42      0.82      0.55      2111
    positive       0.97      0.83      0.89     13847

    accuracy                           0.83     15958
   macro avg       0.69      0.82      0.72     15958
weighted avg       0.90      0.83      0.85     15958



In [7]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

small_train = train.sample(30_000, random_state=1)[["text", "label"]].reset_index(drop=True)
test_df = test[["text", "label"]].reset_index(drop=True)

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=256)

train_ds = Dataset.from_pandas(small_train).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)
print(len(train_ds), len(test_ds))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]

30000 15958


In [1]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(output_dir="out", num_train_epochs=2,
                         per_device_train_batch_size=16, per_device_eval_batch_size=64,
                         learning_rate=2e-5, eval_strategy="epoch", save_strategy="no",
                         fp16=True, report_to="none", logging_steps=200)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'train_ds' is not defined

In [1]:
print("train" in globals(), "test" in globals(), "tok" in globals(), "train_ds" in globals())
!ls /content

False False False False
sample_data


In [2]:
!ls /content

sample_data


In [3]:
!find / -name colab_data.csv 2>/dev/null
!pwd

/content


In [4]:
from google.colab import files
uploaded = files.upload()

Saving colab_data.csv to colab_data.csv


In [5]:
!ls -lh /content

total 62M
-rw-r--r-- 1 root root  62M Oct 14 14:32 colab_data.csv
drwxr-xr-x 1 root root 4.0K Sep 30 13:28 sample_data


In [6]:
!ls -lh /content

total 62M
-rw-r--r-- 1 root root  62M Oct 14 14:32 colab_data.csv
drwxr-xr-x 1 root root 4.0K Sep 30 13:28 sample_data


In [7]:
import torch, pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer

print("GPU available:", torch.cuda.is_available())

df = pd.read_csv("/content/colab_data.csv").dropna(subset=["text"])
train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
print(len(train), len(test))          # must be 63830 15958

small_train = train.sample(30_000, random_state=1)[["text", "label"]].reset_index(drop=True)
test_df = test[["text", "label"]].reset_index(drop=True)

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=256)

train_ds = Dataset.from_pandas(small_train).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)
print(len(train_ds), len(test_ds))    # must be 30000 15958

GPU available: True
63830 15958


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/15958 [00:00<?, ? examples/s]

30000 15958


In [8]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

def metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(output_dir="out", num_train_epochs=2,
                         per_device_train_batch_size=16, per_device_eval_batch_size=64,
                         learning_rate=2e-5, eval_strategy="epoch", save_strategy="no",
                         fp16=True, report_to="none", logging_steps=200)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.152097,0.145169,0.953691,0.893820
2,0.087854,0.167633,0.954631,0.898440


TrainOutput(global_step=3750, training_loss=0.1367962646484375, metrics={'train_runtime': 463.2844, 'train_samples_per_second': 129.51, 'train_steps_per_second': 8.094, 'total_flos': 3962141289863040.0, 'train_loss': 0.1367962646484375, 'epoch': 2.0})

In [9]:
from sklearn.metrics import classification_report

out = trainer.predict(test_ds)
pred = out.predictions.argmax(-1)
print(classification_report(test_df["label"], pred, target_names=["negative", "positive"]))

              precision    recall  f1-score   support

    negative       0.85      0.80      0.82      2111
    positive       0.97      0.98      0.97     13847

    accuracy                           0.95     15958
   macro avg       0.91      0.89      0.90     15958
weighted avg       0.95      0.95      0.95     15958



In [10]:
test_df["pred"] = pred
test_df.to_csv("/content/distilbert_predictions.csv", index=False)

trainer.save_model("/content/distilbert-books")
tok.save_pretrained("/content/distilbert-books")
!zip -qr /content/distilbert-books.zip /content/distilbert-books

from google.colab import files
files.download("/content/distilbert_predictions.csv")
files.download("/content/distilbert-books.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>